In [1]:
import mediapipe as mp
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchinfo  import summary

In [2]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, params):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, params["hidden1"]), nn.GELU(),
            nn.Dropout(params["drop1"]),

            nn.Linear(params["hidden1"], params["hidden2"]), nn.GELU(),
            nn.Dropout(params["drop2"]),

            nn.Linear(params["hidden2"], 32), nn.GELU(),
            nn.Dropout(params["drop3"]),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [3]:
params = {'hidden1': 128, 'hidden2': 64, 'lr': 0.001, 'drop1': 0.5, 'drop2': 0.3, 'drop3': 0.1, 'batch': 64}

model = MLP(126, 26, params)

state = torch.load(
    r"D:\Kuliah\Skripsi\clean\demo_program\best_model.pth",
    map_location=torch.device("cpu")
)

In [4]:
summary(model, input_size=(params['batch'], 126))

Layer (type:depth-idx)                   Output Shape              Param #
MLP                                      [64, 26]                  --
├─Sequential: 1-1                        [64, 26]                  --
│    └─Linear: 2-1                       [64, 128]                 16,256
│    └─GELU: 2-2                         [64, 128]                 --
│    └─Dropout: 2-3                      [64, 128]                 --
│    └─Linear: 2-4                       [64, 64]                  8,256
│    └─GELU: 2-5                         [64, 64]                  --
│    └─Dropout: 2-6                      [64, 64]                  --
│    └─Linear: 2-7                       [64, 32]                  2,080
│    └─GELU: 2-8                         [64, 32]                  --
│    └─Dropout: 2-9                      [64, 32]                  --
│    └─Linear: 2-10                      [64, 26]                  858
Total params: 27,450
Trainable params: 27,450
Non-trainable params: 0
Tota

In [5]:
model.load_state_dict(state)
model.to("cpu")
model.eval()

MLP(
  (net): Sequential(
    (0): Linear(in_features=126, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): GELU(approximate='none')
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): GELU(approximate='none')
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=32, out_features=26, bias=True)
  )
)

In [6]:
def predict_pytorch(model, features):
    x = torch.tensor(features, dtype=torch.float32)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

In [7]:
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=2)

In [8]:
def extract_hands_keypoints(results):
    # Default vector
    output = np.zeros((42, 3))

    if results.multi_hand_world_landmarks and results.multi_handedness:
        for hand_landmarks, handedness in zip(results.multi_hand_world_landmarks, results.multi_handedness):
            idx = 0 if handedness.classification[0].label == 'Left' else 1

            # Ambil koordinat (21x3)
            coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark])

            # Masukkan ke array
            start = idx * 21
            end = start + 21
            output[start:end, :] = coords

    return output

In [9]:
def normalize_keypoints_world(keypoints):
    kp = np.asarray(keypoints, dtype=np.float64)

    left = kp[:21].copy()
    right = kp[21:].copy()

    def norm_hand(hand):
        if np.allclose(hand, 0):
            return hand

        # Translation
        wrist = hand[0]
        h = hand - wrist

        # Scaling
        scale = np.linalg.norm(h[5])
        if scale < 1e-8:
            scale = 1e-8

        h /= scale
        return h

    left_n = norm_hand(left)
    right_n = norm_hand(right)

    return np.vstack([left_n, right_n])

In [10]:
label_classes = [chr(i) for i in range(65, 91)]  # A-Z

In [12]:
cap = cv2.VideoCapture(1)

with mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
) as hands:

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame kosong, skip.")
            break

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(img_rgb)
        keypoints = extract_hands_keypoints(result)
        # print(keypoints)

        if keypoints is not None and np.any(keypoints):

            # Normalisasi → flatten → reshape
            features = normalize_keypoints_world(keypoints).reshape(1, -1)
            
            # Prediksi
            probs = predict_pytorch(model, features)[0]

            pred_idx = np.argmax(probs)
            pred_label = label_classes[pred_idx]
            confidence = probs[pred_idx]
            
            if confidence > 0.7:
                label_text = f"{pred_label} ({confidence*100:.1f}%)"

                cv2.putText(frame, label_text, (30, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)

        # gambar tangan
        if result.multi_hand_landmarks:
            for hand_landmarks in result.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS
                )

        cv2.imshow("Hand Sign Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()